In [ ]:
# ==========================================
# БЛОК ЛОКАЛЬНОГО ТЕСТИРОВАНИЯ
# (Удали этот блок перед отправкой решения!)
# ==========================================
if __name__ == "__main__":
    print("Начинаем локальный тест...")
    from sklearn.metrics import f1_score
    
    # Для теста возьмем последние 1050 строк из train (там есть 7 аномалий)
    # Удаляем колонку 'target', чтобы имитировать тестовые данные
    test_df = train_data.iloc[-1050:].drop(columns=['target']).copy()
    test_features = test_df.values
    
    # Настоящие ответы, с которыми будем сравнивать
    true_target = train_data['target'].iloc[-1050:].values

    # Эмуляция цикла проверяющей системы
    preds = []
    # Нам нужно подавать окно размером 1000. 
    # Всего точек 1050, значит мы можем сделать 50 предсказаний.
    for i in range(len(test_features) - 1000):
        # Подаем окно из 1000 точек
        window_data = test_features[i : 1000 + i + 1].copy()
        prediction = predict(window_data)
        preds.append(prediction)
    
    # Добавляем нули для первых 1000 точек (как сказано в примечании)
    final_preds = np.array([0] * 1000 + preds)
    
    print("Уникальные предсказания:", np.unique(final_preds, return_counts=True))
    
    # Считаем F1 (мы ожидаем увидеть > 0.17)
    score = f1_score(true_target, final_preds)
    print(f"Локальный F1 Score: {score:.4f}")
    print("Тест завершен.")

In [2]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 1. Загрузка данных
train_data = pd.read_csv("train.csv")
num_anomalies = 7

# 2. Функции подготовки признаков
def make_features_numpy(data: np.ndarray) -> np.ndarray:
    shifts = [1, 5, 20, 100]
    all_features = [data]
    for shift in shifts:
        shifted = np.empty_like(data)
        shifted[:shift, :] = 0
        shifted[shift:, :] = data[:-shift, :]
        all_features.append(shifted)
    return np.hstack(all_features)

def make_features_and_target(data: pd.DataFrame) -> np.ndarray:
    data = data.copy()
    target = None
    name_target_col = 'target' 
    if name_target_col in data.columns:
        target = data[name_target_col]
        data = data.drop(columns=[name_target_col])
    return make_features_numpy(data.values), target

# 3. Подготовка обучающей выборки
features, target_series = make_features_and_target(train_data)
target = target_series.values

# Масштабирование
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Поиск аномалий
iso_forest = IsolationForest(contamination=0.002, random_state=42)
iso_forest.fit(features_scaled[:train_data.shape[0]-num_anomalies])
anomaly_score = iso_forest.decision_function(features_scaled).reshape(-1, 1)

X_train_final = np.hstack([features_scaled, anomaly_score])

# Расчет весов для CatBoost
num_normal = train_data.shape[0] - num_anomalies
scale_weight = int(round(num_normal / num_anomalies, 0))

# Обучение основной модели
model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    scale_pos_weight=scale_weight,
    verbose=0,
)
model.fit(X_train_final, target)

# 4. Функция предсказания (Оптимизированная)
def predict(test_data: np.ndarray) -> int:
    last_feature_row = np.hstack([
        test_data[-1],
        test_data[-2],
        test_data[-6],
        test_data[-21],
        test_data[-101]
    ]).reshape(1, -1)
    
    test_scaled = scaler.transform(last_feature_row)
    test_score = iso_forest.decision_function(test_scaled).reshape(1, 1)
    last_row = np.hstack([test_scaled, test_score])
    
    return int(model.predict(last_row)[0])

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'